# FahMai RAG System — Multiple-Choice Q&A
ระบบ Retrieval-Augmented Generation สำหรับตอบคำถามแบบตัวเลือกของร้านฟ้าใหม่

In [1]:
# Cell 1: Install dependencies
!pip install rank_bm25 sentence-transformers faiss-cpu pythainlp openai pandas tqdm numpy --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Cell 2: Imports & Configuration
import os
import re
import time
import json
import glob
import pickle
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from typing import List, Dict
from tqdm import tqdm

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
from pythainlp.tokenize import word_tokenize

# ============================================================
# CONFIGURATION — แก้ไขตรงนี้
# ============================================================
THAILLM_API_KEY = "C8tzmCwayWm67rzXjzgahL45hPem7w2X"  # จาก thaillm.or.th

# ThaiLLM: เลือก 1 model (uncomment ที่ต้องการ)
# THAILLM_MODEL_NAME = "typhoon"       # Typhoon-S 8B by SCB 10X
# THAILLM_MODEL_NAME = "openthaigpt"  # OpenThaiGPT 8B
# THAILLM_MODEL_NAME = "pathumma"     # Pathumma Qwen3-8B
THAILLM_MODEL_NAME = "kbtg"         # THaLLE 8B

EMBED_MODEL_NAME = "BAAI/bge-m3"    
TOP_K_RETRIEVE = 20
TOP_K_CONTEXT = 5
RRF_K = 60
API_SLEEP = 0.3

BASE_DIR = Path(".")
KB_DIR = BASE_DIR / "knowledge_base"
QUESTIONS_PATH = BASE_DIR / "questions.csv"
SUBMISSION_PATH = BASE_DIR / "submission.csv"
CACHE_PATH = BASE_DIR / "_embed_cache.pkl"

print("✅ Imports สำเร็จ")
print(f"🤖 Model: {THAILLM_MODEL_NAME}")
print(f"🔍 Embedding: {EMBED_MODEL_NAME}")
print(f"📁 Knowledge base: {KB_DIR.resolve()}")

✅ Imports สำเร็จ
🤖 Model: typhoon
🔍 Embedding: BAAI/bge-m3
📁 Knowledge base: C:\Users\USER\Desktop\university\superAi\Hack3\super-ai-engineer-s-6-fah-mai-rag-challenge-level-1\data\knowledge_base


In [3]:
# Cell 3: Load Documents from Markdown Files

def load_documents(kb_dir: Path) -> List[Dict]:
    """โหลดไฟล์ markdown ทั้งหมดจาก knowledge base"""
    docs = []
    patterns = [
        kb_dir / "products" / "*.md",
        kb_dir / "policies" / "*.md",
        kb_dir / "store_info" / "*.md",
    ]
    category_map = {"products": "product", "policies": "policy", "store_info": "store_info"}
    
    for pattern in patterns:
        files = sorted(glob.glob(str(pattern)))
        category = pattern.parent.name
        for fpath in files:
            with open(fpath, "r", encoding="utf-8") as f:
                content = f.read()
            filename = Path(fpath).stem
            # ดึง product code จาก filename (เช่น DN-LT-001_daonuea_airbook_15)
            product_code = filename.split("_")[0] if "_" in filename else filename
            docs.append({
                "content": content,
                "source": fpath,
                "filename": filename,
                "category": category_map.get(category, category),
                "product_code": product_code,
            })
    
    print(f"✅ โหลดเอกสารทั้งหมด {len(docs)} ไฟล์")
    for cat in ["product", "policy", "store_info"]:
        count = sum(1 for d in docs if d["category"] == cat)
        print(f"   - {cat}: {count} ไฟล์")
    return docs

documents = load_documents(KB_DIR)

✅ โหลดเอกสารทั้งหมด 118 ไฟล์
   - product: 110 ไฟล์
   - policy: 5 ไฟล์
   - store_info: 3 ไฟล์


In [4]:
# Cell 4: Thai-Aware Section-Based Chunking

def split_by_sections(text: str, max_chars: int = 1200) -> List[str]:
    """
    แบ่ง markdown ตาม header ## / ###
    ถ้า section ยาวเกิน max_chars ให้แบ่งตาม paragraph (\n\n)
    """
    # แบ่งตาม header level 2 และ 3
    header_pattern = re.compile(r'(?m)^(#{1,3} .+)$')
    parts = header_pattern.split(text)
    
    sections = []
    current_header = ""
    current_body = ""
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if header_pattern.match(part):
            # บันทึก section ก่อนหน้า
            if current_body.strip():
                sections.append((current_header, current_body.strip()))
            current_header = part
            current_body = ""
        else:
            current_body += "\n" + part
    
    # บันทึก section สุดท้าย
    if current_body.strip():
        sections.append((current_header, current_body.strip()))
    
    # ถ้าไม่มี header เลย ให้ใช้ทั้ง text
    if not sections:
        sections = [("", text.strip())]
    
    # แบ่งส่วนที่ยาวเกินด้วย paragraph
    final_sections = []
    for header, body in sections:
        if len(body) <= max_chars:
            final_sections.append((header, body))
        else:
            paragraphs = [p.strip() for p in body.split("\n\n") if p.strip()]
            # จัดกลุ่ม paragraphs ให้ไม่เกิน max_chars
            buffer = ""
            for para in paragraphs:
                if len(buffer) + len(para) + 2 <= max_chars:
                    buffer += ("\n\n" if buffer else "") + para
                else:
                    if buffer:
                        final_sections.append((header, buffer))
                    buffer = para
            if buffer:
                final_sections.append((header, buffer))
    
    return final_sections


def chunk_documents(docs: List[Dict]) -> List[Dict]:
    """แบ่ง documents เป็น chunks พร้อม metadata"""
    chunks = []
    
    for doc in docs:
        content = doc["content"]
        
        # ดึง title จากบรรทัดแรก (# ...)
        lines = content.strip().split("\n")
        title = lines[0].lstrip("#").strip() if lines and lines[0].startswith("#") else doc["filename"]
        
        sections = split_by_sections(content)
        
        for i, (section_header, section_body) in enumerate(sections):
            # สร้าง chunk text โดยมี context ของที่มาด้วย
            chunk_parts = []
            chunk_parts.append(f"[ที่มา: {title}]")
            if section_header:
                chunk_parts.append(section_header)
            chunk_parts.append(section_body)
            chunk_text = "\n".join(chunk_parts)
            
            chunks.append({
                "text": chunk_text,
                "source": doc["source"],
                "filename": doc["filename"],
                "category": doc["category"],
                "product_code": doc["product_code"],
                "title": title,
                "section_header": section_header,
                "chunk_index": i,
            })
    
    print(f"✅ สร้าง chunks ทั้งหมด {len(chunks)} chunks")
    print(f"   ความยาวเฉลี่ย: {int(np.mean([len(c['text']) for c in chunks]))} ตัวอักษร/chunk")
    print(f"   ความยาวสูงสุด: {max(len(c['text']) for c in chunks)} ตัวอักษร")
    return chunks


chunks = chunk_documents(documents)
chunk_texts = [c["text"] for c in chunks]

✅ สร้าง chunks ทั้งหมด 888 chunks
   ความยาวเฉลี่ย: 484 ตัวอักษร/chunk
   ความยาวสูงสุด: 1284 ตัวอักษร


In [5]:
# Cell 5: Build BM25 Index with Thai Tokenization

def tokenize_thai(text: str) -> List[str]:
    """Tokenize ภาษาไทยด้วย pythainlp newmm engine"""
    # รวม Thai + English tokens
    tokens = word_tokenize(text, engine="newmm", keep_whitespace=False)
    # lowercase และกรอง token ที่ว่าง
    tokens = [t.lower().strip() for t in tokens if t.strip()]
    return tokens


def build_bm25_index(texts: List[str]) -> BM25Okapi:
    print("🔨 กำลังสร้าง BM25 index...")
    tokenized = [tokenize_thai(t) for t in tqdm(texts, desc="Tokenizing")]
    bm25 = BM25Okapi(tokenized)
    print(f"✅ BM25 index พร้อมแล้ว ({len(texts)} documents)")
    return bm25, tokenized


bm25_index, tokenized_chunks = build_bm25_index(chunk_texts)

🔨 กำลังสร้าง BM25 index...


Tokenizing: 100%|██████████| 888/888 [00:01<00:00, 837.22it/s] 

✅ BM25 index พร้อมแล้ว (888 documents)


In [6]:
# Cell 6: Build Dense Vector Index (FAISS) with Sentence Transformers

def build_dense_index(texts: List[str], model_name: str, cache_path: Path):
    """Encode chunks และสร้าง FAISS index (มี cache เพื่อประหยัดเวลา)"""
    
    # ตรวจสอบ cache
    if cache_path.exists():
        print(f"📂 โหลด embeddings จาก cache: {cache_path}")
        with open(cache_path, "rb") as f:
            cached = pickle.load(f)
        if cached.get("model") == model_name and cached.get("num_chunks") == len(texts):
            embeddings = cached["embeddings"]
            embed_model = SentenceTransformer(model_name)
            print(f"✅ โหลด embeddings จาก cache สำเร็จ shape={embeddings.shape}")
        else:
            print("⚠️  Cache ไม่ตรงกัน จะ encode ใหม่")
            embeddings = None
            embed_model = None
    else:
        embeddings = None
        embed_model = None
    
    if embeddings is None:
        print(f"🔨 กำลัง encode {len(texts)} chunks ด้วย {model_name}...")
        embed_model = SentenceTransformer(model_name)
        embeddings = embed_model.encode(
            texts, 
            batch_size=64, 
            show_progress_bar=True, 
            normalize_embeddings=True  # สำหรับ cosine similarity
        ).astype("float32")
        
        # บันทึก cache
        with open(cache_path, "wb") as f:
            pickle.dump({"model": model_name, "num_chunks": len(texts), "embeddings": embeddings}, f)
        print(f"💾 บันทึก cache ที่ {cache_path}")
    
    # สร้าง FAISS index (Inner Product = Cosine หลัง normalize)
    dim = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(embeddings)
    
    print(f"✅ FAISS index พร้อมแล้ว: {faiss_index.ntotal} vectors, dim={dim}")
    return faiss_index, embed_model, embeddings


faiss_index, embed_model, chunk_embeddings = build_dense_index(
    chunk_texts, EMBED_MODEL_NAME, CACHE_PATH
)

📂 โหลด embeddings จาก cache: _embed_cache.pkl


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ โหลด embeddings จาก cache สำเร็จ shape=(888, 1024)
✅ FAISS index พร้อมแล้ว: 888 vectors, dim=1024


In [7]:
# Cell 7: Hybrid Retrieval with RRF (Reciprocal Rank Fusion)

def bm25_retrieve(query: str, bm25: BM25Okapi, top_k: int) -> List[int]:
    """ค้นหาด้วย BM25 คืน index ของ chunks เรียงตามคะแนน"""
    query_tokens = tokenize_thai(query)
    scores = bm25.get_scores(query_tokens)
    # argsort descending
    ranked = np.argsort(scores)[::-1][:top_k].tolist()
    return ranked


def dense_retrieve(query: str, model: SentenceTransformer, index: faiss.Index, top_k: int) -> List[int]:
    """ค้นหาด้วย Dense Retrieval คืน index ของ chunks"""
    q_embed = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_embed, top_k)
    return indices[0].tolist()


def reciprocal_rank_fusion(ranked_lists: List[List[int]], k: int = 60) -> List[int]:
    """
    Reciprocal Rank Fusion
    score(d) = sum(1 / (k + rank_i(d))) สำหรับทุก ranked list
    """
    scores = {}
    for ranked_list in ranked_lists:
        for rank, doc_idx in enumerate(ranked_list):
            if doc_idx not in scores:
                scores[doc_idx] = 0.0
            scores[doc_idx] += 1.0 / (k + rank + 1)
    
    # เรียงตามคะแนน descending
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_idx for doc_idx, _ in sorted_docs]


def hybrid_retrieve(query: str, top_k_final: int = TOP_K_CONTEXT) -> List[Dict]:
    """
    Hybrid Retrieval: BM25 + Dense + RRF
    คืน list ของ chunks ที่เกี่ยวข้องมากที่สุด
    """
    # Step 1: ดึงผลจาก BM25 และ Dense
    bm25_results = bm25_retrieve(query, bm25_index, TOP_K_RETRIEVE)
    dense_results = dense_retrieve(query, embed_model, faiss_index, TOP_K_RETRIEVE)
    
    # Step 2: RRF fusion
    fused = reciprocal_rank_fusion([bm25_results, dense_results], k=RRF_K)
    
    # Step 3: คืน top-k chunks
    top_indices = fused[:top_k_final]
    return [chunks[i] for i in top_indices]


# ทดสอบ retrieval
test_results = hybrid_retrieve("Watch S3 Ultra กันน้ำได้กี่ ATM")
print("\n🔍 ทดสอบ Hybrid Retrieval:")
for i, r in enumerate(test_results[:3]):
    print(f"  [{i+1}] {r['title']} | {r['section_header'][:50] if r['section_header'] else '-'}")
    print(f"       {r['text'][:150].strip()}...\n")


🔍 ทดสอบ Hybrid Retrieval:
  [1] วงโคจร Watch S3 Pro (WongKhoJon Watch S3 Pro) | ## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
       [ที่มา: วงโคจร Watch S3 Pro (WongKhoJon Watch S3 Pro)]
## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
**Q: Watch S3 Pro มี ECG ไหมครับ? ต่างจาก Watch S3 ยังไง?**...

  [2] วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) | ## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
       [ที่มา: วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra)]
## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
**Q: Watch S3 Ultra ต่างจาก Watch S3 Pro ยังไงบ้างครับ?...

  [3] วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) | ## รายละเอียดสินค้า
       [ที่มา: วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra)]
## รายละเอียดสินค้า
เมื่อกีฬาและเทคโนโลยีมาบรรจบกัน วงโคจร Watch S3 Ultra คือจุดสูงสุดของสา...



In [8]:
# Cell 8: Answer Generation with ThaiLLM API

SYSTEM_PROMPT = """คุณคือ FahMai AI Customer Service ผู้ช่วยตอบคำถามของร้านฟ้าใหม่ ร้านจำหน่ายสินค้าอิเล็กทรอนิกส์ชั้นนำ

กฎการตอบ:
- ตอบในรูปแบบ ANSWER: X เท่านั้น โดย X คือตัวเลข 1-10
- อ่าน Context ที่ให้มาอย่างละเอียดก่อนตอบ
- ถ้าคำตอบอยู่ใน Context → เลือกตัวเลือกที่ถูกต้อง (ANSWER: 1-8)
- ถ้า Context ไม่มีข้อมูลเพียงพอ → ANSWER: 9
- ถ้าคำถามไม่เกี่ยวข้องกับร้านฟ้าใหม่เลย → ANSWER: 10
- ห้ามเดาถ้าไม่มีข้อมูลใน Context

ตอบในรูปแบบ: ANSWER: X"""


def ask_llm(messages, model=THAILLM_MODEL_NAME, max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling."""
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number (1-10) from LLM response."""
    if text is None:
        return 9
    # ลบ <think>...</think> block (pathumma ใช้ chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # หาก ANSWER: X ก่อน
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m and 1 <= int(m.group(1)) <= 10:
        return int(m.group(1))
    # fallback: ตัวเลขแรก 1-10 ในคำตอบ
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    print(f"  ⚠️ Parse ไม่ได้จาก: '{clean[:60]}' → default=9")
    return 9


def build_user_prompt(question: str, choices: List[str], context_chunks: List[Dict]) -> str:
    context_texts = []
    for i, chunk in enumerate(context_chunks):
        context_texts.append(f"--- เอกสาร {i+1} ---\n{chunk['text']}")
    context_str = "\n\n".join(context_texts)

    choices_str = "\n".join([
        f"{i+1}. {choice}"
        for i, choice in enumerate(choices)
        if choice and str(choice).strip() not in ["", "nan"]
    ])

    return f"""Context ที่เกี่ยวข้อง:
{context_str}

---
คำถาม: {question}

ตัวเลือก:
{choices_str}

ตอบในรูปแบบ ANSWER: X (X คือตัวเลข 1-10):"""


def generate_answer(question: str, choices: List[str]) -> int:
    context_chunks = hybrid_retrieve(question, top_k_final=TOP_K_CONTEXT)
    user_prompt = build_user_prompt(question, choices, context_chunks)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    response_text = ask_llm(messages)
    return parse_answer(response_text)


# ทดสอบ API connection
print("🧪 ทดสอบ ThaiLLM API...")
test_resp = ask_llm([{"role": "user", "content": "ตอบในรูปแบบ ANSWER: X → 1+1=2 ถูกหรือผิด?"}], max_retries=1)
if test_resp:
    print(f"✅ API พร้อมใช้งาน! ตอบ: {test_resp[:80]}")
else:
    print("❌ API ยังมีปัญหา ตรวจสอบ API key และ network")

🧪 ทดสอบ ThaiLLM API...
✅ API พร้อมใช้งาน! ตอบ: ANSWER: X → 1+1=2 ถูกหรือผิด?  
**ถูก**  
เพราะ 1 + 1 = 2 เป็นความจริงทางคณิตศาส


In [9]:
# Cell 9: Main Pipeline — Process All 100 Questions

# โหลด questions
df_questions = pd.read_csv(QUESTIONS_PATH)
print(f"📋 โหลดคำถามทั้งหมด {len(df_questions)} ข้อ")
print(f"   Columns: {list(df_questions.columns)}")
print(df_questions.head(2))

# ตรวจสอบ API key
if THAILLM_API_KEY == "ใส่ API key ของคุณตรงนี้":
    raise ValueError("❌ กรุณาตั้งค่า TYPHOON_API_KEY ก่อน! แก้ไขที่ Cell 2 หรือ set environment variable")

# รัน pipeline
results = []
choice_cols = [f"choice_{i}" for i in range(1, 11)]

print(f"\n🚀 เริ่มประมวลผล {len(df_questions)} คำถาม...")
print("-" * 60)

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc="Processing"):
    q_id = int(row["id"])
    question = str(row["question"]).strip()
    choices = [str(row[col]).strip() if col in row and pd.notna(row[col]) else "" for col in choice_cols]
    
    answer = generate_answer(question, choices)
    results.append({"id": q_id, "answer": answer})
    
    # แสดงความคืบหน้า
    if q_id % 10 == 0:
        print(f"  ✓ ข้อ {q_id}/100 → ตอบ: {answer}")
    
    # หน่วงเวลาป้องกัน rate limit
    time.sleep(API_SLEEP)

print("-" * 60)
print(f"✅ ประมวลผลครบ {len(results)} ข้อ")

# แสดงสถิติคำตอบ
df_results = pd.DataFrame(results)
print("\n📊 สถิติคำตอบ:")
print(df_results["answer"].value_counts().sort_index().to_string())

📋 โหลดคำถามทั้งหมด 100 ข้อ
   Columns: ['id', 'question', 'choice_1', 'choice_2', 'choice_3', 'choice_4', 'choice_5', 'choice_6', 'choice_7', 'choice_8', 'choice_9', 'choice_10']
   id                               question   choice_1   choice_2   choice_3  \
0   1   Watch S3 Ultra กันน้ำได้กี่ ATM ครับ      3 ATM       IP68      5 ATM   
1   2  ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ  2,990 บาท  4,490 บาท  1,990 บาท   

    choice_4   choice_5   choice_6   choice_7   choice_8  \
0       IP67     10 ATM     20 ATM       IPX8      1 ATM   
1  4,990 บาท  3,490 บาท  2,490 บาท  3,990 บาท  5,490 บาท   

                    choice_9                                choice_10  
0  ไม่มีข้อมูลนี้ในฐานข้อมูล  คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า  
1  ไม่มีข้อมูลนี้ในฐานข้อมูล  คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า  

🚀 เริ่มประมวลผล 100 คำถาม...
------------------------------------------------------------


Processing:   9%|▉         | 9/100 [00:25<04:17,  2.83s/it]

  ✓ ข้อ 10/100 → ตอบ: 2


Processing:  19%|█▉        | 19/100 [00:54<03:51,  2.86s/it]

  ✓ ข้อ 20/100 → ตอบ: 2


...

  ✓ ข้อ 90/100 → ตอบ: 3


Processing:  99%|█████████▉| 99/100 [04:23<00:04,  4.81s/it]

  ✓ ข้อ 100/100 → ตอบ: 3


Processing: 100%|██████████| 100/100 [04:28<00:00,  2.68s/it]

------------------------------------------------------------
✅ ประมวลผลครบ 100 ข้อ

📊 สถิติคำตอบ:
answer
1     15
2     14
3     11
4     15
5      6
6     10
7      9
8      7
9     12
10     1


In [10]:
# Cell 10: Save submission.csv

df_submission = pd.DataFrame(results)[["id", "answer"]]

# ตรวจสอบความถูกต้อง
assert len(df_submission) == 100, f"❌ ต้องมี 100 rows แต่มี {len(df_submission)}"
assert df_submission["answer"].between(1, 10).all(), "❌ มีคำตอบนอกช่วง 1-10"
assert df_submission["id"].nunique() == 100, "❌ มี id ซ้ำ"

df_submission = df_submission.sort_values("id").reset_index(drop=True)
df_submission.to_csv(SUBMISSION_PATH, index=False)

print(f"✅ บันทึกไฟล์สำเร็จ: {SUBMISSION_PATH.resolve()}")
print(f"   จำนวน rows: {len(df_submission)}")
print(f"   ตัวอย่าง 5 แถวแรก:")
print(df_submission.head(10).to_string(index=False))

print("\n🎉 เสร็จสิ้น! พร้อม submit!")

✅ บันทึกไฟล์สำเร็จ: C:\Users\USER\Desktop\university\superAi\Hack3\super-ai-engineer-s-6-fah-mai-rag-challenge-level-1\data\submission.csv
   จำนวน rows: 100
   ตัวอย่าง 5 แถวแรก:
 id  answer
  1       5
  2       7
  3       2
  4       6
  5       6
  6       8
  7       1
  8       4
  9       4
 10       2

🎉 เสร็จสิ้น! พร้อม submit!


In [11]:
import pandas as pd
import glob
import os


In [12]:
# Load ground truth
ground_truth = pd.read_csv('Hack3_ground_truth.csv')
print(f'Ground truth: {len(ground_truth)} rows')
ground_truth.head()

Ground truth: 100 rows


,id,answer_gt
0,1,5
1,2,7
2,3,2
3,4,6
4,5,6


In [13]:
# Find all submission files
submission_files = [f for f in glob.glob('*.csv') if 'submission' in f.lower()]
submission_files.sort()
print('Submission files found:')
for f in submission_files:
    print(f'  {f}')

Submission files found:
  submission.csv
  submission_typhoon.csv
  submissionv2.csv
  submissionv3.csv
  submissionv4.csv


In [14]:
# Ensure ground truth has expected columns
ground_truth.columns = ground_truth.columns.str.strip()
if "answer_gt" not in ground_truth.columns and "answer" in ground_truth.columns:
    ground_truth = ground_truth.rename(columns={"answer": "answer_gt"})

for filename in submission_files:
    sub = pd.read_csv(filename)
    sub.columns = sub.columns.str.strip()

    if "answer" not in sub.columns and "answer_pred" in sub.columns:
        sub = sub.rename(columns={"answer_pred": "answer"})
    if "answer" not in sub.columns and "answer_gt" in sub.columns:
        sub = sub.rename(columns={"answer_gt": "answer"})

    if not {"id", "answer_gt"}.issubset(ground_truth.columns):
        raise ValueError(f"ground_truth columns: {list(ground_truth.columns)}")
    if not {"id", "answer"}.issubset(sub.columns):
        print(f"skip {filename}, columns={list(sub.columns)}")
        continue

    merged = ground_truth[["id", "answer_gt"]].merge(sub[["id", "answer"]], on="id", how="inner")
    merged["answer_gt"] = pd.to_numeric(merged["answer_gt"], errors="coerce")
    merged["answer"] = pd.to_numeric(merged["answer"], errors="coerce")
    merged = merged.dropna(subset=["answer_gt", "answer"])
    merged["correct"] = merged["answer_gt"].astype(int) == merged["answer"].astype(int)

    wrong = merged[~merged["correct"]]
    print(f"\n--- {filename}: {len(wrong)} wrong answers ---")
    print(wrong[["id", "answer_gt", "answer"]].to_string(index=False) if len(wrong) else "All correct!")



--- submission.csv: 28 wrong answers ---
 id  answer_gt  answer
 10          7       2
 13          7       6
 14          4       3
 24          3       9
 25          5       2
 45          3       2
 60         10       9
 61         10       9
 62         10       9
 76          5       1
 78          3       1
 79          6       4
 80          7       1
 81          6       4
 82          6       9
 83          4       3
 84          5       1
 85          4       2
 86          3       1
 87          8       1
 88          3       4
 90          5       3
 92          4       7
 93          5       1
 94          5       1
 95          4       1
 96          5       3
 98          5       3

--- submission_typhoon.csv: 28 wrong answers ---
 id  answer_gt  answer
 10          7       2
 13          7       6
 14          4       3
 24          3       9
 25          5       2
 45          3       2
 60         10       9
 61         10       9
 62         10       9
 76        